## Python notebook for generating the STAC catalog json and corresponding Item json for Raster & Vector layers

### Tools:
1. Pystac 
2. Rasterio
3. Geopandas
4. Matplotlib

This notebook returns Catalog json for Raster and Vector layers.

### 1. Importing the required modules

In [46]:
import os
import json
import xml.etree.ElementTree as ET
import ee
from datetime import datetime, timezone

import rasterio
from rasterio.warp import transform_bounds
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
from urllib.parse import urlparse
import pystac
import sys
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import constants
import numpy as np
import requests
from shapely.geometry import mapping, box
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension



In [47]:
# Authenticate and Initialize GEE
try:
    ee.Initialize(project='ee-corestackdev')
    print("Google Earth Engine initialized successfully!")
except Exception as e:
    print(f"Initialization failed: {e}")

Google Earth Engine initialized successfully!


### 2. Defining the variables used in the notebook

In [48]:
base_dir="../data/"

blocks_info = [
    {
        "block": "badlapur",
        "location": "uttar_pradesh",
        "admin_boundary_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/admin_boundary_jaunpur_badlapur",
        "nrega_assets_file":"jaunpur_badlapur.geojson",
        "lulc_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m",
        "terrain_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/terrain_raster_jaunpur_badlapur",
        "terrain_vector_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/jaunpur_badlapur_terrain_clusters",
        "clart_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/clart_jaunpur_badlapur",
        "surface_water_bodies_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/swb3_jaunpur_badlapur",
        "drainage_lines_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/drainage_lines_jaunpur_badlapur",
        "change_detection_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/change_jaunpur_badlapur_Afforestation",
        "cropping_intensity_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/cropping_intensity_jaunpur_badlapur_2017-23",
        "tree_health_ccd_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/tree_health_ccd_raster_jaunpur_badlapur_2022",
        "Prec_annual_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/Prec_annual_jaunpur_badlapur",
        "tree_health_ch_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/tree_health_ch_raster_jaunpur_badlapur_2021",
        "tree_health_overall_raster_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/tree_health_overall_change_raster_jaunpur_badlapur",
        "aquifer_vector_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/aquifer_vector_jaunpur_badlapur",
        "soge_vector_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/soge_vector_jaunpur_badlapur",
        "restoration_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/restoration_jaunpur_badlapur_raster",
        "change_detection_raster_CropIntensity_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/change_jaunpur_badlapur_CropIntensity",
        "change_detection_raster_Deforestation_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/change_jaunpur_badlapur_Deforestation",
        "change_detection_raster_Degradation_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/change_jaunpur_badlapur_Degradation",
        "change_detection_raster_Urbanization_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/change_jaunpur_badlapur_Urbanization",
        "drought_frequency_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/drought_jaunpur_badlapur_2017_2022",
        "runoff_annual_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/Runoff_annual_jaunpur_badlapur",
        "well_depth_annual_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/well_depth_net_value_jaunpur_badlapur",
        "deltaG_annual_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/filtered_delta_g_annual_jaunpur_badlapur_uid",
        "deltaG_fortnight_file":"projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/filtered_delta_g_fortnight_jaunpur_badlapur_uid",
        "lulc_raster_asset_id": "projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/jaunpur_badlapur_2017-07-01_2018-06-30_LULCmap_10m",

        "admin_boundary_style_file":"Administrative-Boundary-Style.qml",
        "nrega_assets_style_file":"swb_style.qml",
        "lulc_raster_style_file":"LULC0_12class.qml",
        "terrain_raster_style_file":"terrain_1-12class.qml",
        "terrain_vector_style_file":"Terrain-Vector-Layer-Style.qml",
        "clart_style_file":"CLART-Layer-Style.qml",
        "surface_water_bodies_style_file":"Surface-Waterbody-style.qml",
        "drainage_lines_style_file":"Drainage-Layer-Style.qml",
        "change_detection_raster_style_file":"Afforestation_climate_change.qml",
        "cropping_intensity_style_file":"Cropping_intensity.qml",
        "tree_health_ccd_raster_style_file":"ccd_style.qml",
        "Prec_annual_style_file":"Precipitation_Style.qml",
        "tree_health_ch_raster_style_file":"Tree_health_style_oac.qml",
        "tree_health_overall_raster_style_file":"Tree_health_style_oac.qml",
        "aquifer_vector_style_file":"Aquifer_style.qml",
        "soge_vector_style_file":"SOGE_style.qml",
        "restoration_style_file":"Restoration_style.qml",
        "change_detection_raster_CropIntensity_style_file":"Cropping_Intensity_climate_change.qml",
        "change_detection_raster_Deforestation_style_file":"Deforestation_climate_change.qml",
        "change_detection_raster_Degradation_style_file":"Degradation_climate_change.qml",
        "change_detection_raster_Urbanization_style_file":"Urbanization_climate_change.qml",
        "drought_frequency_style_file":"Drought_style.qml",
        "runoff_annual_style_file":"Runoff_style.qml",
        "well_depth_annual_style_file":"MWS-Well-Depth-18_23.qml",
        "deltaG_annual_style_file":"MWS-Well-Depth-18_23.qml",
        "deltaG_fortnight_style_file":"MWS-Well-Depth-18_23.qml"
    }
]

corestack_dir = os.path.join(base_dir, 'CorestackCatalogs')

### 3. For Raster layers the data range fecthed from filename

In [49]:
def extract_raster_dates_from_filename(raster_filename):
    try:
        print(raster_filename)
        parts = raster_filename.split('_')
        start_date = datetime.strptime(parts[2], "%Y-%m-%d")
        end_date = datetime.strptime(parts[3], "%Y-%m-%d")
        print(start_date)
        print(end_date)
    except Exception as e:
        raise ValueError(f"Failed to extract raster dates from filename '{raster_filename}': {e}")
        
    return start_date, end_date    

### 4. Parsing the QML file for Raster Layers

In [ ]:
def parse_qml_classes(qml_path):
    tree = ET.parse(qml_path)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)

    if not classes:
        for entry in root.findall(".//item"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)

    return classes

### 5. Generating the vector thumbnails from the qml files 

In [ ]:

def _rgb_to_hex(rgb_string):
    if not isinstance(rgb_string, str):
        return None
    try:
        parts = [int(p) for p in rgb_string.split(',')[:3]]
        return f'#{parts[0]:02x}{parts[1]:02x}{parts[2]:02x}'
    except:
        return None

def parse_qml_style(qml_path):
    try:
        tree = ET.parse(qml_path)
        root = tree.getroot()

        fill_color = None
        outline_color = None

        renderer = root.find(".//renderer-v2[@type='singleSymbol']")
        if renderer:
            symbol = renderer.find(".//symbol")
            if symbol:
                fill_color = symbol.get("color")
                outline_color = symbol.get("outlineColor")
        
        if not fill_color and not outline_color:
            renderer = root.find(".//renderer-v2[@type='categorizedSymbol']")
            if renderer:
                symbol_id = renderer.get("symbol")
                if symbol_id:
                    symbol = root.find(f".//symbols/symbol[@name='{symbol_id}']")
                    if symbol:
                        fill_color = symbol.get("color")
                        outline_color = symbol.get("outlineColor")

        if not fill_color and not outline_color:
            for prop in root.findall(".//prop"):
                key = prop.get("key")
                if key == "color":
                    fill_color = prop.get("value")
                elif key == "outline_color":
                    outline_color = prop.get("value")

        fill_color = fill_color if fill_color else '#808080'
        if not fill_color.startswith('#'):
            fill_color = f"#{fill_color}"
        
        outline_color = outline_color if outline_color else '#000000'
        if not outline_color.startswith('#'):
            outline_color = f"#{outline_color}"
            
        return fill_color, outline_color

    except Exception as e:
        print(f"Error parsing QML file: {e}")
        return '#808080', '#000000'

In [53]:
def generate_vector_thumbnail(vector_path, out_path, qml_path):
    
    fill_color, edge_color = parse_qml_style(qml_path)

    if fill_color is None:
        fill_color = "lightblue"
    if edge_color is None:
        edge_color = "blue"

    print(f"Parsed QML fill color: {fill_color}")
    print(f"Parsed QML edge color: {edge_color}")
    
    
    if vector_path.startswith('projects/'):
        
        try:
            fc = ee.FeatureCollection(vector_path)
        except Exception as e:
            print(f"Error accessing GEE asset: {e}")
            return
            
        painted_image = ee.Image.pixelLonLat().paint(fc, 1)

        viz_params = {
            'min': 0,
            'max': 1,
            'bands': 'constant',
            'palette': [fill_color],
        }

        
        thumbnail_url = painted_image.getThumbURL(viz_params)

        print(f"Generated GEE thumbnail URL: {thumbnail_url}")

        
    else:
        
        try:
            gdf = gpd.read_file(vector_path)
        except Exception as e:
            print(f"Error reading vector file: {e}")
            return
        
        
        if gdf.crs is None or gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)

        fig, ax = plt.subplots(figsize=(3, 3))
        fig.patch.set_facecolor("white")
        ax.set_facecolor("white")

        gdf.plot(ax=ax, color=fill_color, edgecolor=edge_color, linewidth=0.5)

        ax.axis('off')

        plt.savefig(out_path, dpi=150, bbox_inches='tight', pad_inches=0, facecolor=fig.get_facecolor())
        plt.close()
        print(f"Generated local thumbnail: {out_path}")

#### 6. Generating the raster thumbnails from the qml files 

In [54]:
def generate_raster_thumbnail(tif_path, out_path, qml_path):
   
    with rasterio.open(tif_path) as src:
        arr = src.read(1) 
        nodata = src.nodata
        if nodata is not None:
            arr = np.ma.masked_equal(arr, nodata)
    
    unique_raster_values = np.unique(arr.compressed() if isinstance(arr, np.ma.MaskedArray) else arr)
    print(f"Unique values in raster data: {unique_raster_values}")
    
    
    style_info = parse_qml_classes(qml_path)

    # Filter QML info to only include values present in the raster data
    filtered_style_info = [cls for cls in style_info if cls.get('value') in unique_raster_values]
    
    values = [cls['value'] for cls in filtered_style_info if 'value' in cls]
    colors = [cls['color'] for cls in filtered_style_info if 'color' in cls]
    
    print(f"Parsed QML values: {values}")
    print(f"Parsed QML colors: {colors}")
    
    
    try:
        if not values or not colors or len(values) != len(colors):
            raise ValueError("Invalid or insufficient palette information in QML file.")
    
        sorted_indices = np.argsort(values)
        sorted_values = np.array(values)[sorted_indices]
        sorted_colors = np.array(colors)[sorted_indices]

        cmap = ListedColormap(sorted_colors)
        bounds = np.array(sorted_values) - 0.5
        bounds = np.append(bounds, sorted_values[-1] + 0.5)
        norm = Normalize(vmin=bounds.min(), vmax=bounds.max())

    except ValueError as e:
        print(f"Skipping palette generation due to error: {e}. Using a default colormap.")
        cmap = 'gray'
        norm = None

    plt.figure(figsize=(3, 3), dpi=100)
    
    plt.imshow(arr, cmap=cmap, norm=norm, interpolation='none')
    plt.axis('off')

    #os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()


### 7. Creating the Raster items and adding the assets

In [55]:
def create_raster_item(location, block, raster_filename, raster_path, block_catalog_dir, raster_thumbnail, raster_style_file, base_dir, data_url, title, stac_output_dir, thumbnail_path, raster_asset_id):
    try:
        data_type = 'unknown'
        nodata = None
        height = None
        width = None
        proj_epsg = None
        bbox = None
        geom = None
        gsd = None

        if raster_asset_id:
            ee_image = ee.Image(raster_asset_id)
            image_info = ee_image.getInfo()
            bands_info = image_info.get('bands', [])
            if bands_info:
                precision = bands_info[0].get('data_type', {}).get('precision')
                if precision == 'float':
                    data_type = 'float32'
                elif precision == 'double':
                    data_type = 'float64'
                elif precision == 'int':
                    data_type = 'int32'
                elif precision == 'byte':
                    data_type = 'uint8'
                else:
                    data_type = precision
                nodata = bands_info[0].get('nodata_value')
            
            bounds_wgs84 = ee_image.geometry().bounds().getInfo()['coordinates']
            bbox = bounds_wgs84[0][0] + bounds_wgs84[0][2]
            geom = ee_image.geometry().bounds().getInfo()
            gsd = ee_image.projection().nominalScale().getInfo()
            print(f"GEE Raster resolution (GSD): {gsd} meters")
            
            qml_classes = parse_qml_classes(raster_style_file)
            
            vis_params = {}
            if qml_classes:
                palette = [cls['color'].replace('#','') for cls in qml_classes]
                values = [cls['value'] for cls in qml_classes]
                vis_params['min'] = min(values)
                vis_params['max'] = max(values)
                vis_params['palette'] = palette
                if bands_info:
                    vis_params['bands'] = bands_info[0]['id']
            else:
                vis_params = {'palette': ['000000']}

            thumbnail_url = ee_image.getThumbURL(vis_params)
            
            response = requests.get(thumbnail_url)
            if response.status_code == 200:
                with open(raster_thumbnail, 'wb') as f:
                    f.write(response.content)
                print(f"Generated GEE thumbnail: {raster_thumbnail}")
            else:
                print(f"Failed to generate GEE thumbnail: {response.status_code}")
            
        elif raster_path:
            with rasterio.open(raster_path) as src:
                bounds = src.bounds
                geom = mapping(box(*bounds))
                data_type = str(src.dtypes[0])
                nodata = src.nodata if src.nodata is not None else 0
                height, width = src.shape
                data_crs = src.crs

                if src.crs and src.crs.is_epsg_code:
                    proj_epsg = src.crs.to_epsg()
                else:
                    proj_epsg = None
                
                if proj_epsg != 32644:
                    reprojected_bounds = transform_bounds(src.crs, 'EPSG:32644', *bounds)
                    bbox = list(reprojected_bounds)
                    gsd_x = (reprojected_bounds[2] - reprojected_bounds[0]) / width
                    gsd_y = (reprojected_bounds[3] - reprojected_bounds[1]) / height
                    gsd = (gsd_x + gsd_y) / 2
                else:
                    bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
                    gsd = src.res[0]
            print(f"Raster resolution (GSD): {gsd} meters")
            
            generate_raster_thumbnail(raster_path, raster_thumbnail, raster_style_file)
            
        start_date, end_date = extract_raster_dates_from_filename(raster_filename=raster_filename)
    except ValueError as e:
        print(f"Warning: Could not extract dates from filename {raster_filename}")
        start_date = constants.DEFAULT_START_DATE
        end_date = constants.DEFAULT_END_DATE
    except Exception as e:
        print(f"Error processing raster file/asset: {e}")
        return None

    style_info = parse_qml_classes(raster_style_file)
    style_json_path = os.path.join(stac_output_dir, os.path.basename(raster_style_file).replace('.qml', '.json'))
    with open(style_json_path, "w") as f:
        json.dump(style_info, f, indent=2)

    item_id = f"{title}_{os.path.splitext(raster_filename)[0]}"    
    
    item = pystac.Item(
        id=item_id,
        bbox=bbox,
        geometry=geom,
        datetime=datetime.now(timezone.utc),
        properties={
            "title": title,
            "description": f"Raster data for {os.path.splitext(raster_filename)[0]} in {block} of {location}",
            "start_datetime": start_date.isoformat() + 'Z',
            "end_datetime": end_date.isoformat() + 'Z',
            "gsd": gsd,
            "gee:asset_id": raster_asset_id,
        }
    )

    if proj_epsg:
        proj_ext = ProjectionExtension.ext(item, add_if_missing=True)
        proj_ext.epsg = proj_epsg
        proj_ext.bbox = bbox
        if height and width:
            proj_ext.shape = [height, width]
        
    if raster_path:
        item.add_asset("data", Asset(
            href=os.path.join(data_url, os.path.relpath(raster_path, start=base_dir)),
            media_type=MediaType.GEOTIFF,
            roles=["data"],
            title="Raster Layer"
        ))
    elif raster_asset_id:
        item.add_asset("data", Asset(
            href=f"https://earthengine.googleapis.com/v1alpha/projects/ee-corestackdev/assets/{raster_asset_id}:getPixels",
            media_type=MediaType.GEOTIFF,
            roles=["data"],
            title="Raster Layer"
        ))
    
    if raster_asset_id:
        item.add_asset("visualizations", Asset(
            href=f"https://code.earthengine.google.com/?asset={raster_asset_id}",
            media_type=MediaType.HTML,
            roles=["visualizations"],
            title="View in GEE Code Editor"
        ))
    
    raster_ext = RasterExtension.ext(item.assets["data"], add_if_missing=True)
    raster_band = RasterBand.create(
        data_type=data_type,
        spatial_resolution=gsd,
        nodata=nodata
    )
    raster_ext.bands = [raster_band]

    classification_ext = ClassificationExtension.ext(item.assets["data"], add_if_missing=True)
    stac_classes = []
    for cls in style_info:
        stac_class_obj = Classification.create(
            value=int(cls["value"]),
            name=cls.get("label") or f"Class {cls['value']}",
            description=cls.get("label"),
            color_hint=cls['color'].replace('#','')
        )
        stac_classes.append(stac_class_obj)
    classification_ext.classes = stac_classes

    item.add_asset("thumbnail", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_thumbnail, start=base_dir)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Raster Thumbnail"
    ))

    item.add_asset("legend", Asset(
        href=os.path.join(data_url, os.path.relpath(style_json_path, start=base_dir)),
        media_type=MediaType.JSON,
        roles=["metadata"],
        title="Legend JSON"
    ))

    item.add_asset("style", Asset(
        href=os.path.join(data_url, os.path.relpath(raster_style_file, start=base_dir)),
        media_type=MediaType.XML,
        roles=["metadata"],
        title="Raster Style (QML)"
    ))

    item.set_self_href(os.path.join(block_catalog_dir, f"{item_id}.json"))
    item.save_object()
    return item

### 8.Fetching the Vector items descriptions from QML and adding the assets

In [56]:
def parse_vector_descriptions(qml_path):
    tree = ET.parse(qml_path)
    root = tree.getroot()
    columns = []
    colnames = []
    coldesc = []
    for entry in root.findall(".//alias"):
        for attr_key, attr_value in entry.attrib.items():
            if (attr_key == 'field'):
                colnames.append(attr_value)
            if (attr_key == 'name'):
                coldesc.append(attr_value)
    vector_desc_df = pd.DataFrame([colnames,coldesc]).T
    vector_desc_df.columns = ['column_name','column_description']
    return vector_desc_df

### 9. Generating the vector items and adding to assets

In [57]:
def create_vector_item(location, block, vector_filename, vector_path, vector_asset_id, vector_desc_df, block_catalog_dir, vector_thumbnail, vector_style_file, base_dir, data_url, title):
    start_date = constants.DEFAULT_START_DATE
    end_date = constants.DEFAULT_END_DATE
    table_columns = []
    geom = None
    bbox = None
    item_id = None
    
    if vector_filename.endswith('.geojson'):
        media_type = MediaType.GEOJSON
    
    if vector_asset_id:
        try:
            ee_feature_collection = ee.FeatureCollection(vector_asset_id)
            unified_geom = ee_feature_collection.geometry()
            bbox = unified_geom.bounds().getInfo()['coordinates'][0]
            geom = unified_geom.getInfo()
            item_id = f"{title}_{vector_asset_id.split('/')[-1]}"
            
            fill_color, edge_color = parse_qml_style(vector_style_file)

            if not fill_color:
                fill_color = '#808080'
            if not edge_color:
                edge_color = '#000000'

            painted_image = ee_feature_collection.style(**{'fillColor': fill_color, 'color': edge_color})
            
            thumbnail_url = painted_image.visualize().getThumbURL({
                'dimensions': '512x512',
                'format': 'png',
                'region': geom
            })
            
            response = requests.get(thumbnail_url)
            if response.status_code == 200:
                with open(vector_thumbnail, 'wb') as f:
                    f.write(response.content)
                print(f"Generated GEE thumbnail: {vector_thumbnail}")
            else:
                print(f"Failed to generate GEE thumbnail: {response.status_code}")
        except Exception as e:
            print(f"Error accessing GEE asset {vector_asset_id}: {e}")
            return None
    else:
        try:
            gdf = gpd.read_file(vector_path)
            gdf_wgs84 = gdf.to_crs(epsg=4326) if gdf.crs is None or gdf.crs.to_epsg() != 4326 else gdf
            bounds = gdf_wgs84.total_bounds
            bbox = [float(b) for b in bounds]
            geom = mapping(gdf_wgs84.unary_union)

            fill_color, outline_color = parse_qml_style(vector_style_file)
            
            generate_vector_thumbnail(vector_path, vector_thumbnail, fill_color, outline_color)

            item_id = f"{title}_{os.path.splitext(vector_filename)[0]}"
            
            vector_merged_df = gdf.dtypes.reset_index()
            vector_merged_df.columns = ['column_name', 'column_dtype']
            vector_merged_df = vector_merged_df.merge(vector_desc_df, on='column_name', how='left').fillna('')
            table_columns = [
                {
                    "name": row['column_name'],
                    "type": str(row['column_dtype']),
                    "description": row['column_description']
                }
                for ind, row in vector_merged_df.iterrows()
            ]
           
        except Exception as e:
            print(f"Error reading local vector file {vector_path}: {e}")
            return None
        
    item = pystac.Item(
        id=item_id,
        geometry=geom,
        bbox=bbox,
        datetime=datetime.now(timezone.utc),
        properties={
            "title": title,
            "description": f"Vector data for {os.path.splitext(vector_filename)[0]} in {block} of {location}",
            "start_datetime": start_date.isoformat() + 'Z',
            "end_datetime": end_date.isoformat() + 'Z',
            "gee:asset_id": vector_asset_id if vector_asset_id else None
        }
    )

    table_ext = TableExtension.ext(item, add_if_missing=True)
    table_ext.columns = table_columns

    if vector_asset_id:
        item.add_asset("data", Asset(
            href=f"https://earthengine.googleapis.com/v1alpha/{vector_asset_id}",
            media_type=MediaType.GEOJSON,
            roles=["data"],
            title="Vector Layer"
        ))
    else:
        item.add_asset("data", Asset(
            href=os.path.join(data_url, os.path.relpath(vector_path, start=base_dir)),
            media_type=media_type,
            roles=["data"],
            title="Vector Layer"
        ))

    if vector_asset_id:
        item.add_asset("visualizations", Asset(
            href=f"https://code.earthengine.google.com/?asset={vector_asset_id}",
            media_type=MediaType.HTML,
            roles=["visualizations"],
            title="View in GEE Code Editor"
        ))

    item.add_asset("thumbnail", Asset(
        href=os.path.join(data_url, os.path.relpath(vector_thumbnail, start=base_dir)),
        media_type=MediaType.PNG,
        roles=["thumbnail"],
        title="Vector Thumbnail"
    ))
    
    item.add_asset("style", Asset(
        href=os.path.join(data_url, os.path.relpath(vector_style_file, start=base_dir)),
        media_type=MediaType.XML,
        roles=["metadata"],
        title="Vector Style"
    ))

    item.set_self_href(os.path.join(block_catalog_dir, f"{item_id}.json"))
    item.save_object()
    return item

### 10. Generating the STAC for each block and iterating the flow for each layer

In [58]:
def generate_stac_for_block(info):
    base_dir = '../data/'
    corestack_dir = os.path.join(base_dir, 'CorestackCatalogs')

    location = info['location']
    block = info['block']
    
    location_dir = os.path.join(corestack_dir, location)
    block_dir=os.path.join(location_dir, block)

    os.makedirs(block_dir, exist_ok=True)
    
    block_catalog = pystac.Catalog(
        id=block,
        title=f"STAC for {block}",
        description=f"STAC catalog for {block} block data in {location}"
    )

    layers_to_process = []
    if 'lulc_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'lulc_raster_file', 'style_key': 'lulc_raster_style_file', 'title':'LULC_raster'})
    if 'raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'raster_file', 'style_key': 'raster_style_file', 'title':'Raster'})
    if 'admin_boundary_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'admin_boundary_file', 'style_key': 'admin_boundary_style_file', 'title': 'admin_boundary'})
    if 'nrega_assets_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'nrega_assets_file', 'style_key': 'nrega_assets_style_file', 'title': 'nrega_assets'})
    if 'vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'vector_file', 'style_key': 'vector_style_file', 'title': 'vector'})
    if 'terrain_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'terrain_raster_file', 'style_key': 'terrain_raster_style_file', 'title': 'terrain_raster'})
    if 'terrain_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'terrain_vector_file', 'style_key': 'terrain_vector_style_file', 'title': 'terrain_vector'})
    if 'clart_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'clart_file', 'style_key': 'clart_style_file', 'title': 'CLART'})
    if 'surface_water_bodies_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'surface_water_bodies_file', 'style_key': 'surface_water_bodies_style_file', 'title': 'Surface_Water_bodies'})
    if 'drainage_lines_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'drainage_lines_file', 'style_key': 'drainage_lines_style_file', 'title': 'Drainage_lines'})             
    if 'change_detection_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_file', 'style_key': 'change_detection_raster_style_file', 'title': 'Change_detection_raster_Afforestation'})
    if 'cropping_intensity_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'cropping_intensity_file', 'style_key': 'cropping_intensity_style_file', 'title': 'Cropping_intensity'})
    if 'tree_health_ccd_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_ccd_raster_file', 'style_key': 'tree_health_ccd_raster_style_file', 'title': 'Tree_health_ccd_raster_2022'})
    if 'Prec_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'Prec_annual_file', 'style_key': 'Prec_annual_style_file', 'title': 'Prec_annual'})
    if 'tree_health_ch_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_ch_raster_file', 'style_key': 'tree_health_ch_raster_style_file', 'title': 'tree_health_ch_raster_2021'})
    if 'tree_health_overall_raster_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'tree_health_overall_raster_file', 'style_key': 'tree_health_overall_raster_style_file', 'title': 'tree_health_overall_raster'})    
    if 'aquifer_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'aquifer_vector_file', 'style_key': 'aquifer_vector_style_file', 'title': 'aquifer_vector'})  
    if 'soge_vector_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'soge_vector_file', 'style_key': 'soge_vector_style_file', 'title': 'soge_vector'})    
    if 'restoration_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'restoration_file', 'style_key': 'restoration_style_file', 'title': 'restoration'})      
    if 'change_detection_raster_CropIntensity_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_CropIntensity_file', 'style_key': 'change_detection_raster_CropIntensity_style_file', 'title': 'change_detection_raster_CropIntensity'})
    if 'change_detection_raster_Deforestation_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Deforestation_file', 'style_key': 'change_detection_raster_Deforestation_style_file', 'title': 'change_detection_raster_Deforestation'})
    if 'change_detection_raster_Degradation_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Degradation_file', 'style_key': 'change_detection_raster_Degradation_style_file', 'title': 'change_detection_raster_Degradation'})
    if 'change_detection_raster_Urbanization_file' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'change_detection_raster_Urbanization_file', 'style_key': 'change_detection_raster_Urbanization_style_file', 'title': 'change_detection_raster_Urbanization'})
    if 'drought_frequency_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'drought_frequency_file', 'style_key': 'drought_frequency_style_file', 'title': 'drought_frequency'})
    if 'runoff_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'runoff_annual_file', 'style_key': 'runoff_annual_style_file', 'title': 'runoff_annual'})
    if 'well_depth_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'well_depth_annual_file', 'style_key': 'well_depth_annual_style_file', 'title': 'well_depth_annual'})    
    if 'deltaG_annual_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'deltaG_annual_file', 'style_key': 'deltaG_annual_style_file', 'title': 'deltaG_annual'})
    if 'deltaG_fortnight_file' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'deltaG_fortnight_file', 'style_key': 'deltaG_fortnight_style_file', 'title': 'deltaG_fortnight'})    
    if 'lulc_raster_asset_id' in info:
        layers_to_process.append({'type': 'raster', 'file_key': 'lulc_raster_asset_id', 'style_key': 'lulc_raster_style_file', 'title': 'lulc_raster_2017'})    
    if 'vector_asset_id' in info:
        layers_to_process.append({'type': 'vector', 'file_key': 'vector_asset_id', 'style_key': 'vector_style_file', 'title': 'vector_asset'})


    for layer in layers_to_process:
        try:
            item = None
            file_key_value = info.get(layer['file_key'])
            style_path = os.path.join(base_dir, info.get(layer['style_key']))
            
            stac_output_dir = os.path.join(base_dir, 'STAC_output')
            os.makedirs(stac_output_dir, exist_ok=True)
            
            if file_key_value and layer['type'] == 'raster':
                title = layer.get('title')
                if 'assets/' in file_key_value:
                    filename_for_thumbnail = file_key_value.split('/')[-1]
                    thumbnail_filename = f'{block}_{filename_for_thumbnail}_thumbnail.png'
                    thumbnail_path = os.path.join(stac_output_dir, thumbnail_filename)
                    item = create_raster_item(
                        location, block, file_key_value, None, block_dir,
                        thumbnail_path, style_path, base_dir, constants.data_url,
                        title, stac_output_dir, thumbnail_path, raster_asset_id=file_key_value
                    )
                else:
                    file_path = os.path.join(base_dir, file_key_value)
                    filename_for_thumbnail = os.path.splitext(file_key_value)[0]
                    thumbnail_filename = f'{block}_{filename_for_thumbnail}_thumbnail.png'
                    thumbnail_path = os.path.join(stac_output_dir, thumbnail_filename)
                    item = create_raster_item(
                        location, block, file_key_value, file_path, block_dir,
                        thumbnail_path, style_path, base_dir, constants.data_url,
                        title, stac_output_dir, thumbnail_path, None
                    )

            elif file_key_value and layer['type'] == 'vector':
                title = layer.get('title')
                
                
                if 'assets/' in file_key_value:
                    file_path = None
                    vector_asset_id = file_key_value
                    filename_for_thumbnail = file_key_value.split('/')[-1]
                else:
                    file_path = os.path.join(base_dir, file_key_value)
                    vector_asset_id = None
                    filename_for_thumbnail = os.path.splitext(file_key_value)[0]

                thumbnail_filename = f'{block}_{filename_for_thumbnail}_thumbnail.png'
                thumbnail_path = os.path.join(stac_output_dir, thumbnail_filename)
                
                vector_desc_df = parse_vector_descriptions(style_path)

                item = create_vector_item(
                    location=location,
                    block=block,
                    vector_filename=file_key_value,
                    vector_path=file_path, 
                    vector_asset_id=vector_asset_id, 
                    vector_desc_df=vector_desc_df,
                    block_catalog_dir=block_dir,
                    vector_thumbnail=thumbnail_path,
                    vector_style_file=style_path,
                    base_dir=base_dir,
                    data_url=constants.data_url,
                    title=title
                )

            if item:
                block_catalog.add_item(item)
            else:
                print(f"Skipping layer '{layer.get('title', 'N/A')}' for block '{block}' as it is not configured correctly.")
        
        except Exception as e:
            print(f"Error processing layer '{layer.get('title', 'N/A')}' for block '{block}': {e}")
            continue 

    block_catalog.set_self_href(os.path.join(block_dir, 'catalog.json'))
    block_catalog.save_object()
    print(f" STAC catalog created for block: {block} in {location}")

    location_catalog_path = os.path.join(location_dir, 'catalog.json')
    location_catalog_modified = False 

    if os.path.exists(location_catalog_path):
        location_catalog = pystac.read_file(location_catalog_path)
        print(f"Loaded existing location catalog: {location}")
    else:
        os.makedirs(location_dir, exist_ok=True)
        location_catalog = pystac.Catalog(
            id=location,
            title=f"STAC for {location}",
            description=f"STAC catalog for data in {location}"
        )
        location_catalog.set_self_href(location_catalog_path)
        print(f"Created new location catalog: {location}")
        location_catalog_modified = True

    child_id_to_add = block_catalog.id
    existing_child_ids = {child.id for child in location_catalog.get_children()} 
    
    if child_id_to_add not in existing_child_ids:
        child_to_add = pystac.read_file(os.path.join(block_dir, 'catalog.json'))
        location_catalog.add_child(child_to_add)
        location_catalog_modified = True 
        print(f"Added block '{block}' to location catalog '{location}'.")
    else:
        print(f"Block '{block}' already exists in location catalog '{location}'") 
    
    if location_catalog_modified:
        location_catalog.normalize_and_save(location_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
        print(f"Updated location catalog for: {location}")

### 11. Generating the root catalog and checking if already location & block exist 

In [59]:
def generate_root_catalog(blocks_info, base_dir, corestack_dir):
    root_catalog_path = os.path.join(corestack_dir, "catalog.json")

    if os.path.exists(root_catalog_path):
        root_catalog = pystac.read_file(root_catalog_path)
        print("Loaded existing root catalog.")
    else:
        root_catalog = pystac.Catalog(
            id="corestack",
            title="CorestackCatalogs",
            description="Root catalog containing all location-based sub-catalogs"
        )
        root_catalog.set_self_href(root_catalog_path) 
        print("Created new root catalog.")
    
    existing_root_children_ids = {child.id for child in root_catalog.get_children()}

    for info in blocks_info:
        location = info["location"]
        location_catalog_path = os.path.join(corestack_dir, location, "catalog.json")

        if os.path.exists(location_catalog_path):
            if location not in existing_root_children_ids:
                location_catalog = pystac.read_file(location_catalog_path)
                root_catalog.add_child(location_catalog)
                existing_root_children_ids.add(location)
                print(f"Added location catalog '{location}' to root catalog.")
            else:
                print(f"Location catalog '{location}' already linked in root catalog.")
        else:
            print(f"Warning: Location catalog not found for {location} at {location_catalog_path}")
                
    root_catalog.set_self_href(os.path.join(corestack_dir, "catalog.json"))
    root_catalog.normalize_and_save(corestack_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    print(f"Root catalog generated at {os.path.join(corestack_dir, 'catalog.json')}")

In [60]:
for block_info in blocks_info:
    print(f"Processing block: {block_info['block']}")
    generate_stac_for_block(block_info)
    
generate_root_catalog(blocks_info, base_dir="../data/", corestack_dir="../data/CorestackCatalogs")

Processing block: badlapur
GEE Raster resolution (GSD): 10 meters
Error processing raster file/asset: Total request size (102750720 bytes) must be less than or equal to 50331648 bytes.
Skipping layer 'LULC_raster' for block 'badlapur' as it is not configured correctly.
Generated GEE thumbnail: ../data/STAC_output/badlapur_admin_boundary_jaunpur_badlapur_thumbnail.png


/tmp/ipykernel_2616033/3793350559.py:51: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geom = mapping(gdf_wgs84.unary_union)


Error reading local vector file ../data/jaunpur_badlapur.geojson: generate_vector_thumbnail() takes 3 positional arguments but 4 were given
Skipping layer 'nrega_assets' for block 'badlapur' as it is not configured correctly.
GEE Raster resolution (GSD): 30.000000000000004 meters
Generated GEE thumbnail: ../data/STAC_output/badlapur_terrain_raster_jaunpur_badlapur_thumbnail.png
projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/terrain_raster_jaunpur_badlapur
Generated GEE thumbnail: ../data/STAC_output/badlapur_jaunpur_badlapur_terrain_clusters_thumbnail.png
GEE Raster resolution (GSD): 30.000000000000004 meters
Generated GEE thumbnail: ../data/STAC_output/badlapur_clart_jaunpur_badlapur_thumbnail.png
projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/clart_jaunpur_badlapur
Generated GEE thumbnail: ../data/STAC_output/badlapur_swb3_jaunpur_badlapur_thumbnail.png
Generated GEE thumbnail: ../data/STAC_output/badlapur_drainage_lines_jaunpur_badla